# Part 3 — Training-Time Optimization Techniques

Five controlled experiments (tensor creation, weight init, activation checkpointing, gradient accumulation, mixed precision) using the same model, data, batch size, and step count throughout, so results are directly comparable.

**Requires PyTorch** (`pip install torch`; a CUDA GPU is needed to observe the memory/speed effects of checkpointing and mixed precision). See `FINDINGS.ipynb` for the expected-direction reference table.

```
Part 3: Training-time Optimization Techniques
==============================================
Covers, each with a runnable snippet AND a controlled experiment using the
SAME model / data / batch size / step count:

  1. Tensor Creation (CPU vs GPU)
  2. Weight Initialization
  3. Activation Checkpointing
  4. Gradient Accumulation
  5. Mixed Precision Training

Environment requirements (NOT available in the sandbox this was written in):
  - PyTorch (`pip install torch`)
  - A CUDA GPU for the GPU-specific comparisons (mixed precision speedups and
    CPU-vs-GPU tensor timing are most meaningful on GPU; the code degrades
    gracefully to CPU-only if no GPU is present, but memory/timing deltas for
    GPU-only techniques won't be observable without one)

Run:
    python part3_optimization_techniques.py
```

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE_GPU = "cuda" if torch.cuda.is_available() else None
DEVICE_CPU = "cpu"

# ---- Fixed experimental setup used across ALL techniques -------------------
IN_DIM, HID_DIM, OUT_DIM = 1024, 2048, 10
BATCH_SIZE = 128
N_STEPS = 50
N_LAYERS = 8
SEED = 42


def make_data(batch_size=BATCH_SIZE, device="cpu"):
    torch.manual_seed(SEED)
    x = torch.randn(batch_size, IN_DIM, device=device)
    y = torch.randint(0, OUT_DIM, (batch_size,), device=device)
    return x, y


class DeepMLP(nn.Module):
    """A deliberately deep MLP so activation checkpointing has something to save memory on."""

    def __init__(self, in_dim=IN_DIM, hid_dim=HID_DIM, out_dim=OUT_DIM, n_layers=N_LAYERS, init="default"):
        super().__init__()
        layers = [nn.Linear(in_dim, hid_dim)]
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hid_dim, hid_dim))
        self.layers = nn.ModuleList(layers)
        self.out = nn.Linear(hid_dim, out_dim)
        self._init_weights(init)

    def _init_weights(self, mode):
        for layer in self.layers + [self.out]:
            if mode == "zeros":
                nn.init.zeros_(layer.weight)
            elif mode == "xavier":
                nn.init.xavier_uniform_(layer.weight)
            elif mode == "kaiming":
                nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
            elif mode == "normal_large":
                nn.init.normal_(layer.weight, mean=0.0, std=1.0)  # deliberately too large
            # "default" -> leave PyTorch's built-in init (kaiming_uniform variant)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

    def forward(self, x, use_checkpoint=False):
        for layer in self.layers:
            if use_checkpoint:
                # Recompute activations on backward instead of storing them.
                # NOTE: `layer=layer` binds the current loop value into the
                # lambda's default argument -- without it, the lambda captures
                # `layer` by reference and by the time backward() re-runs this
                # function, the loop variable has already moved on to a later
                # (differently-shaped) layer, causing a shape mismatch.
                x = torch.utils.checkpoint.checkpoint(
                    lambda inp, layer=layer: F.relu(layer(inp)), x, use_reentrant=False
                )
            else:
                x = F.relu(layer(x))
        return self.out(x)


def gpu_mem_mb():
    if DEVICE_GPU:
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


def reset_gpu_mem():
    if DEVICE_GPU:
        torch.cuda.reset_peak_memory_stats()


def timed_training_loop(model, x, y, device, steps=N_STEPS, use_checkpoint=False,
                         grad_accum_steps=1, use_amp=False):
    """
    Generic training loop reused by every experiment so time/memory/loss are
    measured under identical conditions except for the one technique varied.
    """
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device == "cuda")

    reset_gpu_mem()
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()

    losses = []
    optimizer.zero_grad()
    for step in range(steps):
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                             dtype=torch.float16, enabled=use_amp):
            out = model(x, use_checkpoint=use_checkpoint)
            loss = F.cross_entropy(out, y) / grad_accum_steps

        if use_amp and device == "cuda":
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % grad_accum_steps == 0:
            if use_amp and device == "cuda":
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        losses.append(loss.item() * grad_accum_steps)

    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start
    peak_mem = gpu_mem_mb()
    return {"time_sec": round(elapsed, 4), "peak_gpu_mem_mb": peak_mem, "final_loss": round(losses[-1], 4)}

### 1. Tensor Creation: CPU vs GPU

In [2]:
def experiment_tensor_creation():
    print("\n--- 1. Tensor Creation: CPU vs GPU ---")
    results = {}
    for device in ["cpu"] + (["cuda"] if DEVICE_GPU else []):
        torch.manual_seed(SEED)
        start = time.time()
        for _ in range(1000):
            t = torch.randn(1024, 1024, device=device)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        results[device] = round(elapsed, 4)
        print(f"  1000x create 1024x1024 tensor on {device}: {elapsed:.4f}s")
    if "cuda" not in results:
        print("  [!] No GPU available in this environment; only CPU timing shown.")
    return results


# Snippet reference:
#   cpu_tensor = torch.randn(1024, 1024)                 # created on CPU (default)
#   gpu_tensor = torch.randn(1024, 1024, device="cuda")  # created directly on GPU (faster
#                                                         # than .to("cuda") after CPU creation,
#                                                         # since it skips the host->device copy)

### 2. Weight Initialization

In [3]:
def experiment_weight_init():
    print("\n--- 2. Weight Initialization ---")
    device = DEVICE_GPU or DEVICE_CPU
    x, y = make_data(device=device)
    results = {}
    for mode in ["zeros", "normal_large", "xavier", "kaiming", "default"]:
        model = DeepMLP(init=mode)
        res = timed_training_loop(model, x, y, device)
        results[mode] = res
        print(f"  init={mode:12s} -> {res}")
    return results


# Snippet reference:
#   nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")  # good default for ReLU nets
#   nn.init.xavier_uniform_(layer.weight)                        # good default for tanh/sigmoid nets
#   nn.init.zeros_(layer.weight)                                 # BAD: symmetric, no learning signal

### 3. Activation Checkpointing

In [4]:
def experiment_activation_checkpointing():
    print("\n--- 3. Activation Checkpointing ---")
    device = DEVICE_GPU or DEVICE_CPU
    x, y = make_data(device=device)
    results = {}
    for use_ckpt in [False, True]:
        model = DeepMLP()
        res = timed_training_loop(model, x, y, device, use_checkpoint=use_ckpt)
        results["checkpoint" if use_ckpt else "no_checkpoint"] = res
        print(f"  checkpointing={use_ckpt} -> {res}")
    if not DEVICE_GPU:
        print("  [!] Memory savings from checkpointing are only measurable via CUDA memory stats; "
              "run on a GPU machine to see peak_gpu_mem_mb drop.")
    return results


# Snippet reference:
#   from torch.utils.checkpoint import checkpoint
#   out = checkpoint(layer_fn, x, use_reentrant=False)
#   # Trades compute for memory: activations for `layer_fn` are NOT stored;
#   # they're recomputed during backward(). Time per step goes up slightly,
#   # peak memory goes down -- useful for very deep/large models.

### 4. Gradient Accumulation

In [5]:
def experiment_gradient_accumulation():
    print("\n--- 4. Gradient Accumulation ---")
    device = DEVICE_GPU or DEVICE_CPU
    # Simulate a larger "effective batch" (BATCH_SIZE * accum_steps) without
    # increasing the physical batch that must fit in memory at once.
    results = {}
    for accum_steps in [1, 4, 8]:
        x, y = make_data(batch_size=BATCH_SIZE // accum_steps if accum_steps > 1 else BATCH_SIZE,
                          device=device)
        model = DeepMLP()
        res = timed_training_loop(model, x, y, device, grad_accum_steps=accum_steps)
        results[f"accum_steps={accum_steps}"] = res
        print(f"  accum_steps={accum_steps} (micro-batch={x.shape[0]}) -> {res}")
    return results


# Snippet reference:
#   loss = criterion(model(x), y) / accum_steps
#   loss.backward()
#   if (step + 1) % accum_steps == 0:
#       optimizer.step(); optimizer.zero_grad()
#   # Lets you simulate a large effective batch size on limited GPU memory by
#   # summing gradients over several small "micro-batches" before stepping.

### 5. Mixed Precision Training

In [6]:
def experiment_mixed_precision():
    print("\n--- 5. Mixed Precision Training ---")
    device = DEVICE_GPU or DEVICE_CPU
    x, y = make_data(device=device)
    results = {}
    for use_amp in [False, True]:
        model = DeepMLP()
        res = timed_training_loop(model, x, y, device, use_amp=use_amp)
        results["fp32" if not use_amp else "amp_fp16"] = res
        print(f"  amp={use_amp} -> {res}")
    if not DEVICE_GPU:
        print("  [!] AMP speedups are GPU-specific (uses Tensor Cores); on CPU-only hardware "
              "this mainly demonstrates correctness, not the expected speed/memory gains.")
    return results


# Snippet reference:
#   scaler = torch.cuda.amp.GradScaler()
#   with torch.autocast(device_type="cuda", dtype=torch.float16):
#       loss = criterion(model(x), y)
#   scaler.scale(loss).backward()
#   scaler.step(optimizer); scaler.update()
#   # Runs most ops in fp16 (fast, less memory) while keeping a fp32 master
#   # copy of weights + loss scaling to avoid gradient underflow.


def main():
    print(f"Device available: GPU={'yes (' + torch.cuda.get_device_name(0) + ')' if DEVICE_GPU else 'no'}")
    all_results = {
        "tensor_creation": experiment_tensor_creation(),
        "weight_init": experiment_weight_init(),
        "activation_checkpointing": experiment_activation_checkpointing(),
        "gradient_accumulation": experiment_gradient_accumulation(),
        "mixed_precision": experiment_mixed_precision(),
    }
    print("\n\n=== SUMMARY ===")
    for technique, res in all_results.items():
        print(f"\n{technique}:")
        for k, v in res.items():
            print(f"  {k}: {v}")


if __name__ == "__main__":
    main()

Device available: GPU=yes (Tesla T4)

--- 1. Tensor Creation: CPU vs GPU ---
  1000x create 1024x1024 tensor on cpu: 10.0236s
  1000x create 1024x1024 tensor on cuda: 0.2236s

--- 2. Weight Initialization ---
  init=zeros        -> {'time_sec': 0.5429, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': 2.3001}
  init=normal_large -> {'time_sec': 0.3253, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': nan}
  init=xavier       -> {'time_sec': 0.3787, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': 2.0986}
  init=kaiming      -> {'time_sec': 0.3802, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': 0.0312}
  init=default      -> {'time_sec': 0.3795, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': 2.2997}

--- 3. Activation Checkpointing ---
  checkpointing=False -> {'time_sec': 0.3832, 'peak_gpu_mem_mb': 259.0390625, 'final_loss': 2.2997}
  checkpointing=True -> {'time_sec': 0.6893, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.2998}

--- 4. Gradient Accumulation ---
  accum_steps=1 (micro-batch=128) 

### Run all experiments

In [7]:
main()

Device available: GPU=yes (Tesla T4)

--- 1. Tensor Creation: CPU vs GPU ---
  1000x create 1024x1024 tensor on cpu: 5.4408s
  1000x create 1024x1024 tensor on cuda: 0.0378s

--- 2. Weight Initialization ---
  init=zeros        -> {'time_sec': 0.4109, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.3001}
  init=normal_large -> {'time_sec': 0.3255, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': nan}
  init=xavier       -> {'time_sec': 0.3807, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.0986}
  init=kaiming      -> {'time_sec': 0.3842, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 0.0312}
  init=default      -> {'time_sec': 0.3825, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.2997}

--- 3. Activation Checkpointing ---
  checkpointing=False -> {'time_sec': 0.3856, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.2997}
  checkpointing=True -> {'time_sec': 0.5885, 'peak_gpu_mem_mb': 260.0390625, 'final_loss': 2.2998}

--- 4. Gradient Accumulation ---
  accum_steps=1 (micro-batch=128) -